# YOLOv8 Training on Google Colab
This notebook installs ultralytics, prepares the dataset, and trains a YOLOv8n model.
It saves the best model and prints basic metrics.


In [ ]:
# Install ultralytics
!pip install -q ultralytics


In [ ]:
# Mount Google Drive (optional)
from google.colab import drive
drive.mount('/content/drive')
# Upload a zip of the annotated data if you prefer not to use Drive
%cp /content/drive/MyDrive/data/annotated.zip . || true
!unzip -q annotated.zip -d data || true


In [ ]:
from ultralytics import YOLO
import os
# Verify dataset.yaml exists
if not os.path.exists('data/dataset.yaml'):
    raise FileNotFoundError('data/dataset.yaml not found')
# Load the pretrained YOLOv8n model
model = YOLO('yolov8n.pt')
# Train with the requested hyper‑parameters
results = model.train(
    data='data/dataset.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    augment=True,
    mosaic=1.0,
    mixup=0.2,
    hsv_h=0.015,
    hsv_v=0.4,
    degrees=10,
    device=0
)
# Save the best checkpoint
best_path = getattr(results, 'best', 'runs/detect/train/weights/best.pt')
!cp $best_path ./best.pt
# Download the best model to the local machine
from google.colab import files
files.download('best.pt')
# Print key metrics if they are available
if hasattr(results, 'metrics'):
    metrics = results.metrics
    print('mAP50:', metrics.get('map50', 'N/A'))
    print('Precision:', metrics.get('precision', 'N/A'))
    print('Recall:', metrics.get('recall', 'N/A'))
else:
    print('Metrics not available')
